In [ ]:
import os, json, ast, sys, itertools
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm import tqdm
from matplotlib.patches import Patch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
sys.path.append('..')
from Dataset.index import VOLUMES, VALID, Normalization
from Network.index import ModelNetwork
from Augmentations.index import Augmentations
from Metrics.index import Metrics
from utils.index import faciesCmaps

In [ ]:
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

# MODELO
- Base Path é a pasta dos modelos: '../Model/Backup' ou uma de '../Documents/Backups' (Backup1, Backup2, Backup3)
- O teste e o bloco são sempre do dataset em que o modelo treinou (o `dataset` do `processing`), e as métricas são em relação a ele

In [ ]:
BASE_PATH = '../Model/Backup'

In [ ]:
models = sorted(os.listdir(BASE_PATH), key=lambda name: int(name.split('_')[-1]))
rows   = []

for model in models:
    info     = json.load(open(f'{BASE_PATH}/{model}/info.json'))
    options  = info['processing']
    n_images = info.get('division', options).get('n_images', {})

    rows.append({
        'id': model,
        'network': info['model'].get('network'),
        'dataset': options.get('dataset'),
        'num_filters': options.get('num_filters'),
        'dropout': options.get('dropout'),
        'lr': options.get('lr'),
        'batch_size': options.get('batch_size'),
        'train_size': n_images.get('train'),
        'img_size': info['model'].get('img_size'),
        'test_iou': info['trainer'].get('test_iou'),
        'block_iou': info.get('block_iou', {}).get('mean'),
        'path': f'{BASE_PATH}/{model}'
    })

backups = pd.DataFrame(rows).sort_values(by='dataset')
backups

In [ ]:
modelId = models[-1]    # O ÚLTIMO TREINADO, QUE O Task/index.py AVALIA DEPOIS DE CADA TREINO; OU 'model_1' PARA FIXAR UM
baseDir = f'{BASE_PATH}/{modelId}'

modelInfo = json.load(open(f'{baseDir}/info.json'))
OPTIONS   = modelInfo['processing']
DIVISION  = modelInfo.get('division', OPTIONS)

print(DIVISION)
OPTIONS

In [ ]:
modelOptions = modelInfo['model']
network      = ModelNetwork(**modelOptions)
modelData    = torch.load(f'{baseDir}/model.pth', map_location=network.device)

print(modelOptions)
network.model.load_state_dict(modelData['model'])
network.model.eval()

# SPLITS DOS DADOS

- O `TEST_SIZE` vem do `division` do `info.json` do modelo, gravado pelo `1 - Model` (backup de antes do `division` guardava o `n_images` e o `step` no `processing`, e o notebook lê de lá; o `test_size` era o 0.1 fixo do `1 - Model`): aqui não existe valor próprio, então não tem como treinar com um e testar com outro.
- O split é refeito igual ao do treino e as contagens têm que bater com o `n_images` gravado; se não baterem, a célula para.
- Com `test_size` 0 o modelo treinou só com treino e validação: não há tiles de teste, as seções de teste (métricas, matriz de confusão e predições dos tiles) são puladas e só o bloco é medido.

In [ ]:
DATABASE  = f'../Dataset/{OPTIONS["dataset"]}/DataBase.csv'
df        = pd.read_csv(DATABASE)
IMG_SIZE  = tuple(modelOptions['img_size'])
TILE      = ast.literal_eval(df['shape'].iloc[0])
STEP      = DIVISION.get('step', OPTIONS.get('step'))
SCALING   = OPTIONS.get('scaling', Normalization.DEFAULT)
FORMATTED = Normalization.read(DATABASE)

if FORMATTED != SCALING:
    raise ValueError(f'{DATABASE} está com scaling={FORMATTED!r}, modelo treinou com scaling={SCALING!r}: rode o Format do dataset')

if int(df['step'].iloc[0]) != STEP:
    raise ValueError(f'{DATABASE} foi cortado com step={df["step"].iloc[0]} e o modelo treinou com step={STEP}: rode o Format do dataset')

print(f'{DATABASE}: {len(df)} tiles de shape {df["shape"].iloc[0]}, step {STEP}, janela {IMG_SIZE}')
df

In [ ]:
TEST_SIZE = DIVISION.get('test_size', 0.1)
_, testDf = train_test_split(df, test_size=TEST_SIZE, random_state=42) if TEST_SIZE else (df, df.iloc[:0])
counts    = {'total': len(df), 'test': len(testDf)}

if any(DIVISION['n_images'][key] != counts[key] for key in counts):
    raise ValueError(f'split diferente do treino: {counts} aqui, {DIVISION["n_images"]} no info.json')

counts

# DATASET

- Modelo treinado com `crop` no `augmentations` tem a janela menor que o tile: o tile vai inteiro e a predição é pela janela deslizante do `Augmentations`, com o mesmo `overlap`, `mode` e lote do treino, a mesma conta do `test_iou` gravado.
- Sem `crop`, os blocos são da janela do modelo, como sempre.

In [ ]:
augmentations = Augmentations(OPTIONS.get('augmentations'), batch=OPTIONS.get('batch_size'))
print(f'recorte {augmentations.window}: tile inteiro por janela deslizante (overlap {augmentations.overlap}, {augmentations.mode})' if augmentations.window else f'sem recorte: blocos de {IMG_SIZE}')

In [ ]:
# TILES DE TESTE EM BLOCOS DA JANELA (O ÚLTIMO DE CADA EIXO ENCOSTA NO FIM), OU INTEIROS SEM JANELA
class PredictDataset(Dataset):
    def __init__(self, df, shape, window):
        self.df     = df.reset_index(drop=True)
        self.shape  = tuple(shape)
        self.window = tuple(min(int(w), int(s)) for w, s in zip(window or self.shape, self.shape))
        starts      = [sorted(set(range(0, s - w + 1, w)) | {s - w}) for s, w in zip(self.shape, self.window)]
        self.blocks = list(itertools.product(range(len(self.df)), *starts))

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, index):
        row, d, h, w = self.blocks[index]
        paths        = self.df.iloc[row]
        box          = (slice(d, d + self.window[0]), slice(h, h + self.window[1]), slice(w, w + self.window[2]))
        img          = np.array(np.load(paths.img_path, mmap_mode='r')[box], np.float32)
        mask         = np.array(np.load(paths.mask_path, mmap_mode='r')[box], np.int64)
        return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)


testDataset = PredictDataset(testDf, TILE, None if augmentations.window else IMG_SIZE)
print(f'{len(testDf)} tiles de teste, {len(testDataset)} blocos')

# DATA LOADER

In [ ]:
NUM_WORKERS = 2
BATCH_SIZE  = int(OPTIONS.get('batch_size') or 1)
testLoader  = DataLoader(testDataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
len(testLoader)

# DADOS DE TESTE

- Voxels de todos os tiles de teste somados numa matriz de confusão `classes × classes` (a classe `Metrics` do `Model/Metrics/`) antes de dividir.
- Cada classe é medida contra as outras (`tp` na diagonal, a linha é a classe real e a coluna a predita).
- As métricas são as dos estudos de `Estudos/COMPARACAO.md`, todas por voxel, com o nome por extenso:
  - `accuracy`: acertos / total (a PA de Alaudah et al. (2019) e do CONSS);
  - `class_accuracy` e `mean_class_accuracy`: acurácia de cada classe (o recall, `tp / real`) e a média dela (a MCA);
  - `precision` e `mean_precision`: `tp / predito` e a média;
  - `iou` e `mean_iou`: IoU de cada classe e a média simples (o mIoU);
  - `frequency_weighted_iou`: IoU ponderado pela frequência real de cada classe (o FWIU);
  - `f1` e `mean_f1`: F1 de cada classe, que é o Dice do estudo 3D, e a média simples.
- Vão para o `info.json` do modelo como `test_iou` (IoU de cada classe e `mean`) e `test_metrics` (tudo acima).
- Na tabela, a linha `mean` é a média das classes.
- Com `test_size` 0 não há teste: `test_iou` e `test_metrics` ficam `None` e as células desta seção só avisam.

In [ ]:
# SOMA OS VOXELS DOS TILES NA MATRIZ DE CONFUSÃO, COM A MESMA PREDIÇÃO E O MESMO amp DO TREINO
def evaluate(loader, amp=False):
    metrics = Metrics(network.classes)

    with torch.no_grad(), torch.autocast('cuda', enabled=amp):
        for imgs, masks in tqdm(loader, desc='Avaliando'):
            logits = augmentations.infer(network.model, imgs.to(network.device))
            metrics.update(masks.to(network.device).squeeze(1), torch.argmax(logits, dim=1))

    return metrics


testMetrics = evaluate(testLoader, OPTIONS.get('amp', False)) if len(testDataset) else None
testMetrics.confusion if testMetrics else None

In [ ]:
if testMetrics:
    print(f"teste: {testMetrics} | test_iou gravado no treino {modelInfo['trainer'].get('test_iou'):.4f}")
else:
    print('sem teste (test_size 0): só o bloco é medido')

testMetrics.table() if testMetrics else None

# MATRIZ DE CONFUSÃO

- Em inglês: a linha é a classe real (`Real`), a coluna a predita (`Predicted`), uma linha e uma coluna por classe (`Class 0` … `Class N`).
- A cor e o percentual são a fração da classe real que caiu em cada predita (cada linha soma 100 %); o texto traz também a contagem de voxels.
- A dos tiles de teste vai para `confusion_matrix.png` no backup do modelo; com `test_size` 0 ela não existe. A do bloco inteiro (`block_confusion_matrix.png`) sai na seção do volume.

In [ ]:
# A COR É A FRAÇÃO DA CLASSE REAL (CADA LINHA SOMA 1) E O TEXTO TRAZ TAMBÉM A CONTAGEM DE VOXELS
def plotConfusion(matrix, title=None, save=None):
    n      = len(matrix)
    rates  = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    labels = [f'Class {c}' for c in range(n)]

    fig, ax = plt.subplots(figsize=(max(10, 2.4 * n + 4), max(6, 1.4 * n + 2)))
    fig.colorbar(ax.imshow(rates, cmap='Blues', vmin=0, vmax=1), ax=ax, label='Fraction of the real class')

    for i, j in itertools.product(range(n), range(n)):
        ax.text(j, i, f'{rates[i, j]:.1%}\n{matrix[i, j]:,} voxels', ha='center', va='center', fontsize=9 if n > 2 else 11, color='white' if rates[i, j] > 0.5 else 'black')

    ax.set_xticks(range(n), labels); ax.set_yticks(range(n), labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Real'); ax.set_title(title)

    if save:
        plt.savefig(save, dpi=300, bbox_inches='tight', facecolor='white')

    plt.show()


if testMetrics:
    plotConfusion(testMetrics.confusion, title=f'Confusion Matrix - test tiles ({modelId}, {OPTIONS["dataset"]})', save=os.path.join(baseDir, 'confusion_matrix.png'))
else:
    print('sem teste (test_size 0): sem confusion_matrix.png')

# PREDIÇÕES

In [ ]:
def plotPrediction(index, alpha=0.5):
    img, mask = testDataset[index]

    with torch.no_grad():
        logits = augmentations.infer(network.model, img.unsqueeze(0).to(network.device))

    pred        = torch.argmax(logits, dim=1).squeeze().cpu().numpy()
    img, mask   = img.squeeze().numpy(), mask.squeeze().numpy()
    solid, cmap = faciesCmaps(network.classes)
    center      = [n // 2 for n in img.shape]
    planes      = lambda vol: [vol[center[0], :, :], np.rot90(vol[:, :, center[2]], -1), vol[:, center[1], :]]
    legend      = [Patch(facecolor=solid.colors[c], edgecolor='white', label=f'Classe {c}') for c in range(network.classes)]

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))

    for row, layer in enumerate([mask, pred]):
        for j, (section, labels) in enumerate(zip(planes(img), planes(layer))):
            axes[row, j].imshow(section, cmap='gray')
            axes[row, j].imshow(labels, cmap=cmap, vmin=0, vmax=network.classes - 1, alpha=alpha)
            axes[row, j].axis('off')

    fig.legend(handles=legend, loc='lower center', ncol=min(network.classes, 7), bbox_to_anchor=(0.5, 0.01), fontsize=11)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()


for i in range(min(len(testDataset), 2)):
    plotPrediction(i)

# PREDIÇÃO DO BLOCO INTEIRO

- O bloco é o volume inteiro do dataset em que o modelo treinou (`VOLUMES` do `Dataset/index.py`), predito em blocos do tamanho do tile do `Format` (o `shape` do `DataBase.csv`), pulando os inlines que viraram tile (um bloco a cada `step`): a métrica é só do que a rede não viu.
- Com `SAVE_PRED = True` os inlines de treino também são preditos e o volume completo vai para `predicted.npy` no backup do modelo: uint8 com o mesmo shape e as mesmas classes do rótulo do dataset (no `marlim`, o `npy_multiclass_cut.npy`), sem buraco nenhum. A métrica continua só fora do `step`; os inlines de treino só entram no arquivo.
- Cada bloco passa pela mesma `Normalization` do `Format` (o `scaling` do modelo) e pela predição do treino (`Augmentations.infer`: direta, ou por janela deslizante se o modelo treinou em recorte), com o mesmo `amp`.
- A matriz de confusão do bloco só soma a região do dado real (`VALID` do `Dataset/index.py`): no F3 é o volume `(601, 255, 901)` de Alaudah et al. (2019), sem a borda refletida que veio nos patches; dataset sem entrada mede o volume inteiro.
- Métrica só nos inlines que a rede não viu: o `Format` corta um bloco de `img_size[0]` inlines a cada `step` (o bloco `k` vira tile quando `k % step == 0`, e o split treino/validação/teste é feito só entre esses tiles); aqui o bloco `k` é pulado na matriz pela mesma conta, então treino, validação e teste ficam todos de fora.
- Sai no `info.json` do modelo o `block_iou` (IoU de cada classe, com a classe 0, e o `mean`) e o `block_metrics` (as mesmas métricas do teste, com os mesmos nomes: `accuracy`, `mean_class_accuracy`, `mean_precision`, `mean_iou`, `frequency_weighted_iou`, `mean_f1` e, por classe, `class_accuracy`, `precision`, `iou` e `f1`), e no backup o `block_classes.png` e o `block_confusion_matrix.png`.

In [ ]:
# PREDIZ O VOLUME EM BLOCOS DO TILE E SOMA A MATRIZ DE CONFUSÃO SÓ FORA DO step E NA REGIÃO DO DADO REAL
class PredictBuilder:
    def __init__(self, target_size, classes, normalize, amp=False, valid=None, save_path=None):
        self.size      = tuple(target_size)
        self.classes   = classes
        self.normalize = normalize
        self.amp       = amp
        self.valid     = valid
        self.save_path = save_path

    def update(self):
        tz = self.size[0]
        (i0, i1), (y0, y1), (x0, x1) = self.valid or [(0, n) for n in seismic.shape]
        self.pred    = np.zeros(seismic.shape, dtype=np.uint8)
        self.metrics = Metrics(self.classes)

        with torch.no_grad(), torch.autocast('cuda', enabled=self.amp):
            for z in tqdm(range(0, seismic.shape[0], tz)):
                train = (z // tz) % STEP == 0    # INLINES QUE VIRARAM TILE NO Format

                if train and not self.save_path:
                    continue

                block = self.process(np.asarray(seismic[z:z + tz], dtype=np.float32))
                lo, hi = max(z, i0), min(z + len(block), i1)
                self.pred[z:z + len(block)] = block

                if train or lo >= hi:
                    continue

                self.metrics.update(np.asarray(masks[lo:hi, y0:y1, x0:x1]), block[lo - z:hi - z, y0:y1, x0:x1])

        if self.save_path:
            np.save(self.save_path, self.pred)

    # CLASSES DE UM BLOCO DE INLINES, COM A BORDA REFLETIDA ATÉ MÚLTIPLO DO TILE E CADA TILE PELA Normalization DO Format
    def process(self, src):
        tz, ty, tx = self.size
        pads       = [(0, -n % s) for n, s in zip(src.shape, self.size)]
        img        = np.pad(src, pads, mode='reflect') if any(p for _, p in pads) else src
        result     = np.zeros(img.shape, dtype=np.uint8)

        for y, x in itertools.product(range(0, img.shape[1], ty), range(0, img.shape[2], tx)):
            tile   = np.ascontiguousarray(self.normalize(img[:, y:y + ty, x:x + tx]), dtype=np.float32)
            logits = augmentations.infer(network.model, torch.from_numpy(tile)[None, None].to(network.device))
            result[:, y:y + ty, x:x + tx] = torch.argmax(logits, dim=1)[0].cpu().numpy()

        return result[:src.shape[0], :src.shape[1], :src.shape[2]]


BLOCK     = OPTIONS['dataset']
RAW_DIR   = f'../Dataset/{BLOCK}/files/'
SAVE_PRED = False    # TAMBÉM OS INLINES DE TREINO, NO predicted.npy DO BACKUP (~3,5 GB NO marlim); A MÉTRICA NÃO MUDA

seismic, masks = [np.load(os.path.join(RAW_DIR, name), mmap_mode='r') for name in VOLUMES[BLOCK]]
savePath       = os.path.join(baseDir, 'predicted.npy') if SAVE_PRED else None

builder = PredictBuilder(TILE, network.classes, Normalization(SCALING), OPTIONS.get('amp', False), VALID.get(BLOCK), savePath)
builder.update()

print(f'bloco {BLOCK} {seismic.shape}, região medida {builder.valid or "o volume inteiro"}: {builder.metrics.total:,} voxels fora do step')
print(f'predicted.npy: {savePath} {builder.pred.shape} {builder.pred.dtype}, igual ao rótulo {masks.shape} {masks.dtype}' if SAVE_PRED else 'sem predicted.npy')
builder.pred.shape

In [ ]:
from utils.Plotter.index import Plotter


# O info.json NA ORDEM trainer, processing, division, model, test_iou, test_metrics, block_iou, block_metrics, SEM O iou ANTIGO
blockIoU  = builder.metrics.iou()
modelInfo = {key: value for key, value in modelInfo.items() if key not in ('iou', 'test_iou', 'test_metrics', 'block_iou', 'block_metrics')}
modelInfo.update({
    'test_iou': testMetrics.iou() if testMetrics else None,
    'test_metrics': testMetrics.summary() if testMetrics else None,
    'block_iou': blockIoU,
    'block_metrics': builder.metrics.summary()
})

with open(f'{baseDir}/info.json', 'w', encoding='utf-8') as file:
    json.dump(modelInfo, file, ensure_ascii=False, indent=4)

plt.figure(figsize=(17, 6))
Plotter(blockIoU, title=f'IoU por classe no bloco fora do step - {modelId} ({modelInfo["model"].get("network")}, {BLOCK})')
plt.savefig(os.path.join(baseDir, 'block_classes.png'))
plt.show()

In [ ]:
print(f'bloco {BLOCK} fora do step: {builder.metrics}')
builder.metrics.table()

In [ ]:
plotConfusion(builder.metrics.confusion, title=f'Block Confusion Matrix - out of step ({modelId}, {BLOCK})', save=os.path.join(baseDir, 'block_confusion_matrix.png'))

# COMPARAÇÃO VISUAL

In [ ]:
# NA PREDIÇÃO A LEGENDA TRAZ O IoU DE CADA CLASSE E O TÍTULO O mIoU, OS DOIS NO BLOCO FORA DO STEP
def plotViews(img, mask, pred, alpha=0.5, iou=None, saveDir=None):
    mx          = img.shape[0] // 2
    solid, cmap = faciesCmaps(network.classes)

    for name, layer in [('sismica', None), ('real', mask), ('pred', pred)]:
        scores = iou if name == 'pred' and iou else {}
        plt.figure(figsize=(20, 10))
        plt.imshow(np.array(img[mx, :, :]), cmap='gray')

        if layer is not None:
            labels = [f'Classe {c} ({scores[str(c)] * 100:.2f}% IoU)'.replace('.', ',') if str(c) in scores else f'Classe {c}' for c in range(network.classes)]
            plt.imshow(np.array(layer[mx, :, :]), cmap=cmap, vmin=0, vmax=network.classes - 1, alpha=alpha)
            plt.legend(handles=[Patch(facecolor=solid.colors[c], edgecolor='white', label=labels[c]) for c in range(network.classes)], loc='lower right', fontsize=12)

        miou = f" - mIoU {scores['mean'] * 100:.2f}% fora do step".replace('.', ',') if 'mean' in scores else ''
        plt.title(f'{name.capitalize()} - Slice X={mx}{miou}'); plt.axis('off')

        if saveDir:
            plt.savefig(os.path.join(saveDir, f'volume_{name}.png'), dpi=200, bbox_inches='tight')

        plt.show()


viewsDir = os.path.join(baseDir, 'views')
os.makedirs(viewsDir, exist_ok=True)

tz = TILE[0]
ix = (seismic.shape[0] // 2) // tz * tz
ix = ix + tz if (ix // tz) % STEP == 0 else ix    # FATIA DO MEIO QUE A REDE NÃO VIU (SEM SAVE_PRED AS DE TREINO FICAM SEM PREDIÇÃO)
plotViews(np.asarray(seismic[ix:ix + tz], dtype=np.float32), np.asarray(masks[ix:ix + tz], dtype=np.uint8), builder.pred[ix:ix + tz], iou=blockIoU, saveDir=viewsDir)